### Pipeline 
Review → DistilBERT → Emotion Detection
        → Confidence Scores
        → Dashboard Visualization

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)
import re
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import classification_report

from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

SEED = 42


d:\vscode\E-Commerce-Review-Intelligence-System\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Real Dataset (GoEmotions)

In [54]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")

train_data = dataset["train"]
val_data = dataset["validation"]

In [30]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [42]:
train_data['labels']

Column([[27], [27], [2], [14], [3], ...])

### Prepare Multi-Label Data

GoEmotions is multi-label, so we convert labels → multi-hot vectors.

In [55]:
import numpy as np

num_labels = 28  # GoEmotions total labels

def encode_labels(example):
    labels = np.zeros(num_labels, dtype=np.float32)
    for l in example["labels"]:
        labels[l] = 1.0
    example["labels"] = labels.tolist()
    return example

train_data = train_data.map(encode_labels)
val_data = val_data.map(encode_labels)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map: 100%|██████████| 5426/5426 [00:00<00:00, 14682.34 examples/s]


In [56]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1851.88it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Tokenization

In [57]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
val_data = val_data.map(tokenize, batched=True)

# Ensure labels are float for BCEWithLogitsLoss in multi-label setup
train_data = train_data.map(lambda x: {"labels": np.array(x["labels"], dtype=np.float32)})
val_data = val_data.map(lambda x: {"labels": np.array(x["labels"], dtype=np.float32)})

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5426/5426 [00:00<00:00, 9668.08 examples/s] 


### Training

In [59]:
from transformers import Trainer, TrainingArguments, default_data_collator

def multilabel_data_collator(features):
    batch = default_data_collator(features)
    batch["labels"] = batch["labels"].float()  # BCEWithLogitsLoss expects float targets
    return batch

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=multilabel_data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Inference with Confidence Scores

In [60]:
import torch
import torch.nn.functional as F

label_names = dataset["train"].features["labels"].feature.names

def predict_emotions(text, threshold=0.3):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    results = {}
    for i, p in enumerate(probs):
        if p > threshold:
            results[label_names[i]] = float(p)

    return results

In [61]:
predict_emotions("I am really खुश but also nervous about results")

{'admiration': 0.4629044830799103,
 'amusement': 0.48018890619277954,
 'anger': 0.44960153102874756,
 'annoyance': 0.46745243668556213,
 'approval': 0.4789540469646454,
 'caring': 0.47235003113746643,
 'confusion': 0.4850000739097595,
 'curiosity': 0.48656165599823,
 'desire': 0.4863174259662628,
 'disappointment': 0.46448662877082825,
 'disapproval': 0.5109712481498718,
 'disgust': 0.4545260965824127,
 'embarrassment': 0.4732950031757355,
 'excitement': 0.44717276096343994,
 'fear': 0.4649129807949066,
 'gratitude': 0.4472413957118988,
 'grief': 0.48977988958358765,
 'joy': 0.42672091722488403,
 'love': 0.4457746148109436,
 'nervousness': 0.4462786018848419,
 'optimism': 0.4585007131099701,
 'pride': 0.44980520009994507,
 'realization': 0.44500482082366943,
 'relief': 0.49164003133773804,
 'remorse': 0.4566699266433716,
 'sadness': 0.5441082715988159,
 'surprise': 0.5039207339286804,
 'neutral': 0.4983906149864197}